# ElasticNet Regression: Combining L1 and L2 Penalties

ElasticNet combines the benefits of Ridge (L2) and Lasso (L1) regularization for improved model performance with grouped features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# Create synthetic dataset with grouped correlated features
np.random.seed(42)
n_samples, n_features = 300, 100
X = np.random.randn(n_samples, n_features)

# Introduce correlation structure (grouped features)
for i in range(0, n_features, 10):
    for j in range(i+1, min(i+10, n_features)):
        X[:, j] = 0.8 * X[:, i] + 0.2 * np.random.randn(n_samples)

# True coefficients (sparse)
true_coef = np.zeros(n_features)
true_coef[np.random.choice(n_features, 15, replace=False)] = np.random.randn(15) * 3
y = X @ true_coef + 0.5 * np.random.randn(n_samples)

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
# ElasticNetCV for optimal hyperparameters
alphas = np.logspace(-4, 1, 50)
l1_ratios = np.linspace(0, 1, 11)

elasticnet_cv = ElasticNetCV(alphas=alphas, l1_ratio=l1_ratios, cv=5, random_state=42, max_iter=10000)
elasticnet_cv.fit(X_train, y_train)

print(f"Optimal alpha: {elasticnet_cv.alpha_:.6f}")
print(f"Optimal l1_ratio: {elasticnet_cv.l1_ratio_:.4f}")
print(f"Non-zero coefficients: {np.sum(elasticnet_cv.coef_ != 0)}")
print(f"Train R² score: {elasticnet_cv.score(X_train, y_train):.4f}")
print(f"Test R² score: {elasticnet_cv.score(X_test, y_test):.4f}")

# Compare with Ridge and Lasso
from sklearn.linear_model import Ridge, Lasso

ridge = Ridge(alpha=0.1)
ridge.fit(X_train, y_train)

lasso = Lasso(alpha=0.01)
lasso.fit(X_train, y_train)

# Comparison
print("\nModel Comparison:")
print(f"ElasticNet Test R²: {elasticnet_cv.score(X_test, y_test):.4f}")
print(f"Ridge Test R²: {ridge.score(X_test, y_test):.4f}")
print(f"Lasso Test R²: {lasso.score(X_test, y_test):.4f}")

# Visualize coefficient distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

models = [elasticnet_cv, ridge, lasso]
titles = ['ElasticNet', 'Ridge', 'Lasso']
colors = ['steelblue', 'green', 'orange']

for ax, model, title, color in zip(axes, models, titles, colors):
    ax.bar(range(len(model.coef_)), model.coef_, alpha=0.7, color=color)
    ax.set_xlabel('Feature Index')
    ax.set_ylabel('Coefficient Value')
    ax.set_title(f'{title} Coefficients')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Elastic Net Regression

Elastic Net combines L1 and L2 regularization for better feature selection and stability.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
# Create correlated features dataset
np.random.seed(456)
n_samples = 200
X = np.random.randn(n_samples, 10)

# Add correlated features
X[:, 5] = X[:, 0] + 0.3 * np.random.randn(n_samples)
X[:, 6] = X[:, 1] + 0.2 * np.random.randn(n_samples)

# Target with specific pattern
coef = np.array([3, -2, 1.5, 0, 0, 2, -1, 0, 0, 0.5])
y = X @ coef + 0.1 * np.random.randn(n_samples)

print(f"Dataset shape: {X.shape}")
print(f"Correlation between X0 and X5: {np.corrcoef(X[:, 0], X[:, 5])[0, 1]:.3f}")